# Clustering Analysis Using Synthetic and Real-World Data

**Data Science for Health Informatics — Homework 2**

This notebook applies unsupervised learning techniques to two datasets. First, clustering is explored using a synthetic `make_moons()` dataset. Three algorithms are compared: K-Means with 2 clusters, K-Means with 5 clusters, and DBSCAN. The analysis is then applied to the UCI Adult Census Income dataset using two numerical features: age and hours worked per week.


In [ ]:
student_name = "Chantale NzeggeMvele"
student_id = ""  # Student ID removed before public GitHub publication
student_background = "health"


## 1. Synthetic Dataset

A synthetic dataset with 100 observations is generated using `make_moons()` from scikit-learn. The two-dimensional dataset contains two known groups. A small amount of noise is introduced to make the clustering task more realistic. The true labels returned by `make_moons()` are retained so that the estimated clusters can be visually compared with the known structure.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

X, y = make_moons(
    n_samples=100,
    noise=0.1,
    random_state=42
)

df_moons = pd.DataFrame(
    X,
    columns=["feature_1", "feature_2"]
)
df_moons["true_label"] = y

df_moons.head()


### Inspecting the data range

The two numerical features have different ranges. Standardization is therefore useful when applying distance-based algorithms such as K-Means. The standardized values are used for the clustering experiments below.


In [ ]:
print(df_moons[["feature_1", "feature_2"]].describe())


In [ ]:
scaler_moons = StandardScaler()
X_scaled = scaler_moons.fit_transform(X)

df_moons_scaled = pd.DataFrame(
    X_scaled,
    columns=["feature_1", "feature_2"]
)
df_moons_scaled["true_label"] = y

df_moons_scaled.describe()


### Visualizing the synthetic dataset

The scatterplot shows the characteristic two-moon structure. The true labels provide the reference against which the clustering algorithms can be compared.


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y)
plt.title("Synthetic Make-Moons Dataset")
plt.xlabel("Standardized Feature 1")
plt.ylabel("Standardized Feature 2")
plt.grid(True)
plt.show()


### Calculating the real cluster centers

The real cluster centers are calculated from the known labels returned by `make_moons()`. These centers are descriptive averages of the observations belonging to each true group.


In [ ]:
def compute_cluster_centers(X, labels):
    unique_labels = np.unique(labels)
    centers = np.array([
        X[labels == label].mean(axis=0)
        for label in unique_labels
    ])
    return centers

real_centers = compute_cluster_centers(X_scaled, y)
real_centers


### Visualization function

The following function is used to display cluster assignments and, when available, cluster centers. It is used consistently for the real clusters and the clustering results.


In [ ]:
def visualize_clustering_algorithm(X, labels, centers=None, title=""):
    plt.figure(figsize=(7, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, s=30)

    if centers is not None:
        plt.scatter(
            centers[:, 0],
            centers[:, 1],
            marker="x",
            s=150,
            label="Cluster centers"
        )
        plt.legend()

    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.grid(True)
    plt.show()

visualize_clustering_algorithm(
    X_scaled,
    y,
    centers=real_centers,
    title="Real Clusters with Real Centers"
)


## 2. Clustering the Synthetic Dataset

Three clustering approaches are evaluated:

1. K-Means with 2 clusters
2. K-Means with 5 clusters
3. DBSCAN

K-Means is expected to have difficulty with the non-convex moon shapes because it partitions observations around centroids. DBSCAN is included because it can identify clusters based on density and does not require the number of clusters to be specified in advance.


### K-Means with 2 clusters

K-Means is configured with two clusters to match the number of known groups in the synthetic dataset.


In [ ]:
kmeans_2 = KMeans(
    n_clusters=2,
    random_state=42
)

labels_kmeans_2 = kmeans_2.fit_predict(X_scaled)

visualize_clustering_algorithm(
    X_scaled,
    labels_kmeans_2,
    centers=kmeans_2.cluster_centers_,
    title="K-Means with 2 Clusters"
)


### Visual analysis — K-Means with 2 clusters

**Are the estimated clusters equal to the real clusters?**  
Not exactly. K-Means roughly separates the two moon-shaped groups, but its centroid-based partition does not follow the curved structure of the moons.

**Are the estimated centers close to the real centers?**  
The estimated centers are reasonably close to the corresponding descriptive centers, although they do not capture the curved geometry of the groups.

**Do you see a problem caused by bad initialization?**  
No major initialization problem is apparent. K-Means uses k-means++ initialization by default, and `random_state=42` makes the result reproducible.

**Would the resulting clusters be useful?**  
They provide a useful baseline, but K-Means is not an ideal method for these non-convex clusters. A density-based method such as DBSCAN is more appropriate for the observed shape.


### K-Means with 5 clusters

The same K-Means approach is now run with five clusters, even though the synthetic data contains two true groups. This demonstrates how changing K affects the resulting partition.


In [ ]:
kmeans_5 = KMeans(
    n_clusters=5,
    random_state=42
)

labels_kmeans_5 = kmeans_5.fit_predict(X_scaled)

visualize_clustering_algorithm(
    X_scaled,
    labels_kmeans_5,
    centers=kmeans_5.cluster_centers_,
    title="K-Means with 5 Clusters"
)


### Visual analysis — K-Means with 5 clusters

**Are the estimated clusters equal to the real clusters?**  
No. The two moon-shaped groups are divided into several smaller regions rather than being recovered as the two underlying groups.

**Are the estimated centers close to the real centers?**  
The five centroids describe local regions of the feature space rather than the two real groups, so they are not directly comparable with the two true centers.

**Do you see a problem caused by bad initialization?**  
There is no clear evidence of a bad initialization. The main issue is the selected value of K, which is five although the known data contains two groups.

**Would the resulting clusters be useful?**  
The result is not useful for recovering the known two-group structure. It illustrates that selecting an inappropriate number of clusters can lead to over-segmentation.


### DBSCAN

DBSCAN is used as the third clustering method. It is suitable for this example because it can identify non-linear, density-connected structures without requiring the number of clusters to be specified in advance.


In [ ]:
dbscan = DBSCAN(
    eps=0.3,
    min_samples=5
)

labels_dbscan = dbscan.fit_predict(X_scaled)

visualize_clustering_algorithm(
    X_scaled,
    labels_dbscan,
    title="DBSCAN Clustering"
)


### Visual analysis — DBSCAN

**Are the estimated clusters equal to the real clusters?**  
The DBSCAN result is much closer to the two moon-shaped groups than the K-Means results.

**Are the estimated centers close to the real centers?**  
Cluster centers are not a natural output of DBSCAN, so the centroid comparison used for K-Means is not directly applicable.

**Do you see a problem caused by bad initialization?**  
No. DBSCAN does not use centroid initialization. Its result is determined by the selected density parameters and the data.

**Would the resulting clusters be useful?**  
Yes, for this synthetic example DBSCAN is a better fit to the natural structure. However, the `eps` and `min_samples` parameters should still be evaluated when applying DBSCAN to new datasets.


## 3. Clustering the UCI Adult Dataset

The second part of the homework applies K-Means to the Adult Census Income dataset used in Homework 1. Two numerical features are selected: **age** and **hours per week**.

Only these two variables are used for clustering. This makes it possible to visualize the resulting clusters directly in two dimensions, but it also means that the clusters represent patterns in these two variables only and not the full Adult dataset.


### Loading and preprocessing the Adult dataset

The Adult training data is loaded from the UCI Machine Learning Repository. Missing values represented by `?` are converted to missing values and removed. The two selected numerical features are then extracted for clustering.


In [ ]:
adult_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

adult_columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

df_adult = pd.read_csv(
    adult_url,
    header=None,
    names=adult_columns,
    na_values=" ?",
    skipinitialspace=True
)

df_adult_clean = df_adult.dropna().copy()

df_adult_clean["age"] = pd.to_numeric(
    df_adult_clean["age"],
    errors="coerce"
)
df_adult_clean["hours_per_week"] = pd.to_numeric(
    df_adult_clean["hours_per_week"],
    errors="coerce"
)

features = df_adult_clean[
    ["age", "hours_per_week"]
].dropna().copy()

features.reset_index(drop=True, inplace=True)

print("Observations used for clustering:", len(features))
features.head()


### Normalization

K-Means uses distances to assign observations to clusters. Age and hours worked per week are measured on different ranges, so standardization is applied before clustering. This gives both selected features a comparable scale.


In [ ]:
scaler_adult = StandardScaler()

features_scaled = scaler_adult.fit_transform(features)

features_scaled_df = pd.DataFrame(
    features_scaled,
    columns=["age", "hours_per_week"]
)

features_scaled_df.describe()


## 4. Experimental Evaluation — Selecting K

Silhouette score is calculated for K values from 2 through 10. A higher score indicates that observations are, on average, better separated from observations in other clusters and closer to observations in their own cluster.


In [ ]:
silhouette_scores = []
k_values = range(2, 11)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42
    )
    labels = kmeans.fit_predict(features_scaled)
    score = silhouette_score(features_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}, Silhouette Score={score:.4f}")


The original analysis produced the following scores:

- K=2: 0.3677
- K=3: 0.3796
- K=4: 0.4242
- K=5: 0.3866
- K=6: 0.3980
- K=7: 0.4068
- K=8: 0.4262
- K=9: 0.4153
- K=10: 0.3962

Among the tested values, **K=8 has the highest silhouette score (0.4262)**. K=4 is very close (0.4242), so the difference is small and should be considered when interpreting the result.


### Silhouette score versus number of clusters


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)
plt.title("Silhouette Score vs. Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()


## 5. Final K-Means Model

Based on the tested values, **K=8** gives the highest silhouette score. The model is therefore fitted again using eight clusters. The result is visualized using the two standardized features.


In [ ]:
best_k = k_values[np.argmax(silhouette_scores)]

kmeans_final = KMeans(
    n_clusters=best_k,
    random_state=42
)

cluster_labels = kmeans_final.fit_predict(features_scaled)

print("Selected K:", best_k)
print("Silhouette score:", silhouette_scores[best_k - 2])


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    features_scaled[:, 0],
    features_scaled[:, 1],
    c=cluster_labels,
    s=30
)

plt.scatter(
    kmeans_final.cluster_centers_[:, 0],
    kmeans_final.cluster_centers_[:, 1],
    marker="x",
    s=150,
    label="Centroids"
)

plt.title(f"K-Means Clustering of Adult Dataset (K={best_k})")
plt.xlabel("Standardized Age")
plt.ylabel("Standardized Hours per Week")
plt.legend()
plt.grid(True)
plt.show()


## 6. Numeric Analysis of the Adult Dataset

### Which number of clusters is best?

Based on the silhouette-score experiment, **K=8** is the best value among the tested values because it produced the highest score, **0.4262**. However, the score for K=4 was almost identical at 0.4242. Therefore, K=8 is the numerical optimum in this experiment, but the difference is small.


### How well did K-Means cluster the groups?

K-Means produced a **moderate** clustering result using only age and hours worked per week. The best silhouette score of 0.4262 suggests some separation between clusters, but the score is not strong enough to conclude that the population naturally contains eight distinct groups. Additional features and alternative clustering methods would be useful for further analysis.


### Would I deploy this clustering model as a final solution?

I would **not deploy this model as a final solution**. It was developed using only two numerical variables, and the difference between the best and fourth-best K values is very small. Before deployment, I would test additional clinically or contextually meaningful variables, alternative algorithms, stability across samples, and whether the resulting clusters have a meaningful interpretation.


## 7. Conclusion

The synthetic example demonstrated why the choice of clustering algorithm should reflect the structure of the data. K-Means can struggle with non-convex shapes, while DBSCAN was better suited to the moon-shaped dataset.

For the Adult dataset, K-Means was evaluated for K values from 2 to 10 using the silhouette score. K=8 gave the highest score among the tested values, but the moderate score and the small difference from K=4 indicate that further analysis would be necessary before treating these clusters as meaningful population groups.
